In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

OUT_DIR = "sources/imagenes"
CANDIDATES_CSV = OUT_DIR + "/candidatas_wikimedia.csv"
GPKG = "data/catastro_manzana.gpkg"

API = "https://commons.wikimedia.org/w/api.php"
# Wikimedia asks every script to identify itself (bot policy):
HEADERS = {"User-Agent": "ColegioImperial3D/0.1 (research project; https://github.com/vegafabianr/colegio-imperial-madrid-3d)"}
PAUSE = 1.0   # Seconds between requests, to be polite with the server

CATEGORIES = [
    "Category:Exterior of the Colegiata de San Isidro",
    "Category:Instituto San Isidro",
    "Category:Colegio Imperial de San Pedro y San Pablo",
]
SUBCATEGORY_DEPTH = 1   # Also look one level inside each category

# Geographic search around the block (center in EPSG:25830, radius in meters):
CENTER_UTM = (440070, 4473770)
RADIUS = 150

# Categories that we do not want (interiors and old art):
EXCLUDE_WORDS = ["interior", "painting", "engraving", "in art", "retablo", "altar", "chapel", "capilla",
                 "capella", "nef", "choeur", "transept", "cripta", "crypt", "sepulcro", "navidad"]
MIN_YEAR = 2010   # The model shows the building today
MAX_DIST = 30     # Photos found only by position: camera at most this far from the block (meters)

In [ ]:
# Celda 2
import os
import time
import csv
import requests
import shapely
import geopandas as gpd
from pyproj import Transformer

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Celda 3
# One call to the Commons API (with the pause):
def ask(params):
    params["format"] = "json"
    answer = requests.get(API, params=params, headers=HEADERS, timeout=60)
    answer.raise_for_status()
    time.sleep(PAUSE)
    return answer.json()

In [ ]:
# Celda 4
# Files and subcategories of one category:
def category_members(category):
    files = []
    subcategories = []
    params = {"action": "query", "list": "categorymembers", "cmtitle": category, "cmlimit": "500", "cmtype": "file|subcat"}
    while True:
        data = ask(dict(params))
        for member in data["query"]["categorymembers"]:
            if member["ns"] == 6:
                files.append(member["title"])
            elif member["ns"] == 14:
                subcategories.append(member["title"])
        if "continue" not in data:
            break
        params["cmcontinue"] = data["continue"]["cmcontinue"]
    return files, subcategories

In [ ]:
# Celda 5
# Collect file titles from the categories (and their subcategories, except the excluded ones):
def is_excluded(title):
    lower = title.lower()
    for word in EXCLUDE_WORDS:
        if word in lower:
            return True
    return False

titles = set()
pending = [(category, 0) for category in CATEGORIES]
visited = set()
while pending:
    category, depth = pending.pop()
    if category in visited or is_excluded(category):
        continue
    visited.add(category)
    files, subcategories = category_members(category)
    titles.update(files)
    print(len(files), "files in", category)
    if depth < SUBCATEGORY_DEPTH:
        for sub in subcategories:
            pending.append((sub, depth + 1))

category_titles = set(titles)
print(len(titles), "files from categories")

In [ ]:
# Celda 6
# Add the files with coordinates near the block:
to_latlon = Transformer.from_crs(25830, 4326, always_xy=True)
lon, lat = to_latlon.transform(*CENTER_UTM)
data = ask({"action": "query", "list": "geosearch", "gscoord": str(lat) + "|" + str(lon),
            "gsradius": str(RADIUS), "gsnamespace": "6", "gslimit": "500"})
near_titles = [item["title"] for item in data["query"]["geosearch"]]
print(len(near_titles), "files within", RADIUS, "m")
titles.update(near_titles)
titles = sorted(titles)
print(len(titles), "files in total")

In [ ]:
# Celda 7
# Metadata of each file (50 per request): size, author, license, date, camera position:
def clean(value):
    # "<a href=...>Juan Pérez</a>" -> "Juan Pérez"
    text = ""
    inside_tag = False
    for char in str(value):
        if char == "<":
            inside_tag = True
        elif char == ">":
            inside_tag = False
        elif not inside_tag:
            text += char
    return text.strip()

rows = []
for start in range(0, len(titles), 50):
    batch = titles[start:start + 50]
    data = ask({"action": "query", "titles": "|".join(batch), "prop": "imageinfo|coordinates",
                "iiprop": "url|size|mime|extmetadata", "colimit": "500"})
    for page in data["query"]["pages"].values():
        if "imageinfo" not in page:
            continue
        info = page["imageinfo"][0]
        meta = info.get("extmetadata", {})
        coords = page.get("coordinates", [{}])[0]
        rows.append({
            "title": page["title"],
            "mime": info["mime"],
            "width": info["width"],
            "height": info["height"],
            "mb": round(info["size"] / 1e6, 2),
            "date": clean(meta.get("DateTimeOriginal", {}).get("value", ""))[:10],
            "author": clean(meta.get("Artist", {}).get("value", ""))[:80],
            "license": meta.get("LicenseShortName", {}).get("value", ""),
            "lat": coords.get("lat", ""),
            "lon": coords.get("lon", ""),
            "url": info["url"],
            "page": info["descriptionurl"],
        })
print(len(rows), "files with metadata")

In [ ]:
# Celda 8
# Year of a date text: first group of 4 digits between 1500 and 2100.
# "Taken on 2019-05-03 10:21" -> 2019 ; "circa 1640" -> 1640 ; "" -> None
def find_year(text):
    for i in range(len(text) - 3):
        piece = text[i:i + 4]
        if piece.isdigit() and 1500 <= int(piece) <= 2100:
            return int(piece)
    return None

# Distance (meters) from the camera to the block, when the file has coordinates:
parcels = gpd.read_file(GPKG, layer="parcelas")
block = parcels.geometry.union_all()
to_utm = Transformer.from_crs(4326, 25830, always_xy=True)

for row in rows:
    row["year"] = find_year(row["date"])
    row["from_category"] = row["title"] in category_titles
    if row["lat"] != "":
        x, y = to_utm.transform(row["lon"], row["lat"])
        row["dist_m"] = round(block.distance(shapely.Point(x, y)))
    else:
        row["dist_m"] = ""

In [ ]:
# Celda 9
# Keep photos (JPEG) from MIN_YEAR on, not from excluded topics, and
# (found in a category) or (camera within MAX_DIST meters of the block):
selected = []
for row in rows:
    if row["mime"] != "image/jpeg":
        continue
    if is_excluded(row["title"]):
        continue
    if row["year"] is None or row["year"] < MIN_YEAR:
        continue
    near = row["dist_m"] != "" and row["dist_m"] <= MAX_DIST
    if not row["from_category"] and not near:
        continue
    selected.append(row)

selected.sort(key=lambda row: row["title"])
print(len(selected), "candidate photos |", round(sum(r["mb"] for r in selected), 1), "MB in total")

In [ ]:
# Celda 10
# Save the list of candidates (nothing is downloaded yet):
with open(CANDIDATES_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(selected[0].keys()))
    writer.writeheader()
    writer.writerows(selected)
print("Saved:", CANDIDATES_CSV)

licenses = {}
for row in selected:
    licenses[row["license"]] = licenses.get(row["license"], 0) + 1
print(licenses)

In [ ]:
# Celda 11
# Photos chosen by hand from the candidates (exteriors of the Colegiata and the Instituto, and the streets around):
DOWNLOAD_DIR = OUT_DIR + "/wikimedia"
CHOSEN = [
    "File:2013-04 Spain 049 (14231235581).jpg",
    "File:Balcón de la Colegiata de San Isidro.JPG",
    "File:Calle de Toledo (Madrid) 02.jpg",
    "File:Calle de la Colegiata (Madrid) 01.jpg",
    "File:Colegiata de San Isidro (Madrid) (33158106133).jpg",
    "File:Colegiata de San Isidro (Madrid) 10.jpg",
    "File:Colegiata de San Isidro (Madrid) 11.jpg",
    "File:Colegiata de San Isidro (Madrid) 12.jpg",
    "File:Colegiata de San Isidro (Madrid) 13.jpg",
    "File:Colegiata de San Isidro (Madrid) 14.jpg",
    "File:Colegiata de San Isidro (Madrid) 15.jpg",
    "File:Colegio Imperial - Escudo.jpg",
    "File:Iglesia de San Isidro - 01.jpg",
    "File:Iglesia de San Isidro Madrid.jpg",
    "File:Iglesia san Isidro Madrid.JPG",
    "File:Instituto de San Isidro (Madrid) 01 var.jpg",
    "File:Instituto de San Isidro (Madrid) 01.jpg",
    "File:Madrid - Calle de Toledo con la Colegiata de San Isidro al fondo. Madrid. - 130209 181718.jpg",
    "File:Madrid - Colegiata San Isidro 12.JPG",
    "File:Madrid - Colegiata San Isidro 13.JPG",
    "File:Madrid - Colegiata San Isidro 14.JPG",
    "File:Madrid - Colegiata San Isidro 15.JPG",
    "File:Madrid - Colegiata de San Isidro - 130209 181713.jpg",
    "File:Madrid - Colegiata de San Isidro - 130209 182048.jpg",
    "File:Main facade of the Colegiata de San Isidro 2022.jpg",
]
chosen_rows = [row for row in selected if row["title"] in CHOSEN]
print(len(chosen_rows), "of", len(CHOSEN), "chosen photos found |", round(sum(r["mb"] for r in chosen_rows), 1), "MB")

In [ ]:
# Celda 12
# Download the chosen photos one by one (skip the ones already on disk), and keep author and license:
# "File:Colegiata de San Isidro (Madrid) 10.jpg" -> "Colegiata de San Isidro (Madrid) 10.jpg"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
for row in chosen_rows:
    file_name = row["title"][len("File:"):].replace("/", "_")
    path = DOWNLOAD_DIR + "/" + file_name
    row["local_file"] = path
    if os.path.exists(path):
        continue
    answer = requests.get(row["url"], headers=HEADERS, timeout=300)
    answer.raise_for_status()
    with open(path, "wb") as f:
        f.write(answer.content)
    time.sleep(PAUSE)
print(len(os.listdir(DOWNLOAD_DIR)), "files in", DOWNLOAD_DIR)

# Attribution list (needed by the CC BY / CC BY-SA licenses):
with open(DOWNLOAD_DIR + "/atribuciones.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["local_file", "title", "author", "license", "year", "page"])
    writer.writeheader()
    for row in chosen_rows:
        writer.writerow({name: row[name] for name in ["local_file", "title", "author", "license", "year", "page"]})
print("Saved:", DOWNLOAD_DIR + "/atribuciones.csv")